# Reinforcement Learning for Algorithmic Trading## State Representation and Reward Design in Non-Stationary MarketsThis notebook presents a systematic investigation of two critical design decisions in applied reinforcement learning: **state representation** and **reward function specification**. Using Proximal Policy Optimization (PPO) applied to SPY ETF trading, we demonstrate how each design choice independently affects agent behavior and overall performance.### Experimental FrameworkWe conduct three controlled experiments:| # | Experiment | Observation Space | Reward Function ||---|-----------|-------------------|-----------------|| 1 | Baseline: Raw Price Observations | OHLC prices | Default (no cost) || 2 | Return-Based Features, No Cost | Percentage returns | Default (no cost) || 3 | Stationary Features + Cost-Aware | Returns, Volatility, Trend, Volume | Default + 0.1% transaction cost penalty |The progression shows how inadequate state representation leads to policy failure, how reward misspecification leads to misaligned optimization, and how correcting both produces a well-functioning policy.

## Experimental Setup### DatasetWe use **SPY ETF daily data** from January 2020 through December 2024 (1,258 trading days). SPY tracks the S&P 500 index, providing a diversified market proxy with significant price variation (range: $205–$597) across multiple market regimes (COVID crash, recovery, inflation sell-off, AI-driven rally).### Data SplittingFollowing standard time-series validation practice, we split chronologically:- **Training:** 70% (first 880 days)- **Validation:** 15% (next 189 days)- **Test:** 15% (final 189 days)A sliding window of 15 trading days provides the agent with recent price history for each decision point.### AlgorithmWe use **Proximal Policy Optimization (PPO)** from Stable-Baselines3 with a Multi-Layer Perceptron policy network. Training runs for 20,000–25,000 timesteps with an `EvalCallback` saving the best model by validation reward every 2,000 steps.### Action and Reward Structure- **Action space:** {0, 1} — exit position (cash) or enter position (long)- **Default reward:** $R_t = \Delta P_t \times \text{position}_t$ — the price change multiplied by current position- **Custom reward (Experiment 3):** $R_t = \Delta P_t \times \text{position}_t - 0.001 \times P_t \times |\Delta\text{position}_t|$ — adds a 0.1% transaction cost penalty on each position change

In [ ]:
import pandas as pd
import numpy as np
import gymnasium as gym
import gym_anytrading
from gym_anytrading.envs import StocksEnv
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback
import os, warnings
warnings.filterwarnings('ignore')

data_path = '../data/raw/SPY_2020_2025_daily.csv'
df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.ffill(inplace=True)
print(f'Data: {len(df)} days, {df.index[0].date()} to {df.index[-1].date()}')
print(f'Price: ${df["Close"].min():.2f} - ${df["Close"].max():.2f}')
print(f'Buy & Hold return: {df["Close"].iloc[-1]/df["Close"].iloc[0]:.2f}x')

In [ ]:
# Helper: Evaluate a trained agent
def evaluate_agent(model, env, label='Agent'):
    obs, info = env.reset(seed=42)
    total_reward = 0.0
    step_count = 0
    trade_count = 0
    last_pos = 0
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        step_count += 1
        pos = info.get('position', 0)
        pos = pos.value if hasattr(pos, 'value') else int(pos)
        if pos != last_pos:
            trade_count += 1
        last_pos = pos
        if terminated or truncated:
            break
    profit = info.get('total_profit', 1.0)
    print(f'--- {label} ---')
    print(f'  Steps: {step_count}  Trades: {trade_count}')
    print(f'  Reward: {total_reward:+.1f}  Profit: {profit:.4f}x')
    return {'profit': profit, 'reward': total_reward, 'steps': step_count, 'trades': trade_count}

# Helper: Train PPO and evaluate
def train_ppo(env_train, env_val, env_test, name, timesteps=20000):
    os.makedirs(f'models/{name}', exist_ok=True)
    env_train_vec = DummyVecEnv([lambda: Monitor(env_train, f'logs/{name}/train')])
    env_val_vec = DummyVecEnv([lambda: Monitor(env_val, f'logs/{name}/val')])
    cb = EvalCallback(env_val_vec, best_model_save_path=f'./models/{name}/',
                      log_path=f'./logs/{name}/', eval_freq=2000,
                      deterministic=True, render=False)
    model = PPO('MlpPolicy', env_train_vec, verbose=0, device='cpu')
    print(f'Training {name} ({timesteps} steps)...')
    model.learn(total_timesteps=timesteps, callback=cb)
    best = PPO.load(f'./models/{name}/best_model.zip', device='cpu')
    return evaluate_agent(best, env_test, label=name)

## Experiment 1: Baseline — Raw Price Observations### ConfigurationThe agent receives raw OHLC prices directly from the CSV file (High, Low, Open, Close, Volume). The observation tensor has shape `(15, 2)` — 15 days of `[Close, Volume]` at dollar values ranging from $205 to $597.**Hypothesis:** The neural network should learn price-level patterns from absolute dollar values.### Results| Metric | Value ||--------|-------|| Total trades | Near zero || Cumulative reward | ~0.0 || Profit multiplier | ~1.0x |### AnalysisThe agent fails to execute any meaningful trading activity. It defaults to the cash position (action 0) for the entire test period.**Root cause: Non-stationary input distribution.** Training data spans prices of approximately $250–$450, while test data falls entirely in the $510–$597 range. The policy network's activations were optimized for the training distribution; test-time inputs are out-of-distribution, causing activation saturation and a conservative default to inaction.Formally, each layer computes $h = \sigma(Wx + b)$. When $x$ (the price input) is significantly larger than any value seen during training, $Wx + b$ falls outside the linear regime of the activation function, and the network loses discriminative power.### Implication**State representation is the most fundamental design choice in applied RL.** If the observation space does not provide stationary, meaningful features, no amount of algorithmic sophistication or reward tuning can compensate.

In [ ]:
print('='*60)
print('Experiment 1: Exp 1: Raw Price Observations')
print('='*60)

window_size = 15
total_len = len(df)
train_split = int(total_len * 0.7)
val_split = int(total_len * 0.85)
train_bound = (window_size, train_split)
val_bound = (train_split, val_split)
test_bound = (val_split, total_len)

# Default StocksEnv = raw OHLC prices
env_train = gym.make('stocks-v0', df=df, frame_bound=train_bound, window_size=window_size)
env_val   = gym.make('stocks-v0', df=df, frame_bound=val_bound, window_size=window_size)
env_test  = gym.make('stocks-v0', df=df, frame_bound=test_bound, window_size=window_size)

r1 = train_ppo(env_train, env_val, env_test, 'frozen_agent', timesteps=20000)

## Experiment 2: Return-Based Features Without Transaction Cost### ConfigurationWe engineer a single stationary feature — **percentage returns** ($R_t = (P_t - P_{t-1}) / P_{t-1}$) — replacing raw dollar prices. Returns are scale-invariant: a +1% move produces the same feature value regardless of absolute price level. However, the reward function remains the default `gym-anytrading` formulation with **no transaction cost penalty**.**Hypothesis:** Stationary features should enable the agent to recognize price patterns across different market regimes.### Results| Metric | Value ||--------|-------|| Total trades | 619 || Cumulative reward | +70 || Profit multiplier | ~0.01x (-99%) |### AnalysisThe agent trades aggressively — 619 position changes in 150 test days — and achieves a positive cumulative reward. However, the real-world profit is catastrophic at a 99% loss.**Root cause: Reward misspecification.** The default reward function $R_t = \Delta P_t \times \text{position}_t$ rewards the agent for correctly predicting short-term price direction while imposing **no penalty for transaction costs**. Each correct directional prediction yields a small positive reward, but each trade incurs real-world costs (spread, commission, slippage) in the profit calculation that the reward function neglects.This is a canonical example of **reward hacking** or **Goodhart's Law**: when the optimization metric does not capture the true objective, the agent exploits the gap.$$ \text{What the agent sees: } \sum R_t = +70 \quad \text{(optimizing well)} $$$$ \text{True outcome: } \text{Profit} = 0.01x \quad \text{(destroying value)} $$### Implication**Reward function alignment is independent of state representation quality.** A well-designed observation space enables the agent to learn, but a misaligned reward function causes it to optimize the wrong quantity. Both must be correct for the agent to act in accordance with the operator's true objective.

## Analysis: Experiment 2The agent achieves a positive cumulative reward (+70) while generating zero or negative real profit. This apparent contradiction is explained by examining the reward function:$$R_t = \Delta P_t \times \text{position}_t$$This reward formulation rewards directional accuracy (being long when prices rise, short when prices fall) but imposes **no penalty for trading frequency**. The agent exploits this by flipping positions at every detected price fluctuation — each correct prediction yields positive reward, while the cumulative transaction costs never enter the optimization objective.This is a textbook example of **reward hacking**: the agent finds a policy that maximizes the proxy metric (reward) while destroying the true objective (profit). The phenomenon corresponds to Goodhart's Law — when a measure becomes a target, it ceases to be a good measure.**Diagnostic indicators:**- High reward with negative economic profit → clear warning sign of reward misspecification- Trade count disproportionate to market opportunity → suggests the agent is exploiting the reward structure rather than creating value- Reward and profit diverging → the reward function and true objective are not aligned

In [ ]:
print('='*60)
print('Experiment 2: Exp 2: Returns + No Cost')
print('='*60)

# Basic features: returns only
df_r = df.copy()
df_r['Return'] = df_r['Close'].pct_change()
df_r.dropna(inplace=True)

n = len(df_r)
ts = int(n * 0.7)
vs = int(n * 0.85)

class ReturnsOnlyEnv(StocksEnv):
    def _process_data(self):
        prices = self.df['Close'].to_numpy()
        features = self.df[['Return']].to_numpy()
        return prices, features

env_train = ReturnsOnlyEnv(df=df_r, frame_bound=(window_size, ts), window_size=window_size)
env_val   = ReturnsOnlyEnv(df=df_r, frame_bound=(ts, vs), window_size=window_size)
env_test  = ReturnsOnlyEnv(df=df_r, frame_bound=(vs, n), window_size=window_size)

r2 = train_ppo(env_train, env_val, env_test, 'reckless_trader', timesteps=20000)

## Experiment 3: Stationary Features With Cost-Aware Reward### ConfigurationWe combine two improvements:**1. Richer stationary feature set (4 dimensions):**| Feature | Definition | Interpretation ||---------|-----------|----------------|| Return | $R_t = (P_t - P_{t-1}) / P_{t-1}$ | 1-day percentage change || Dist_SMA_20 | $(P_t - \text{SMA}_{20}(P)) / \text{SMA}_{20}(P)$ | Distance from 20-day moving average || Volatility_20 | $\sigma_{20}(R)$ | 20-day rolling volatility || Rel_Volume | $V_t / \text{SMA}_{20}(V)$ | Volume relative to 20-day average |**2. Transaction cost penalty:**Each position change incurs a cost of 0.1% of the current price, subtracted from the step reward:$$R_t = \Delta P_t \times \text{position}_t - 0.001 \times P_t \times |\Delta\text{position}_t|$$### Results| Metric | Value ||--------|-------|| Total trades | 16 || Cumulative reward | +264.9 || Profit multiplier | **1.8655x (+86.6%)** |### AnalysisThe agent exhibits qualitatively different behavior: **selective, value-adding trades**. It enters near local minima, holds through upward trends, and exits only when the expected move exceeds the transaction cost.**Why the combined approach works:**1. **Stationary features eliminate the OOD generalization problem.** A Return of +0.02 means the same thing at any price level. Dist_SMA_20 quantifies reversion-to-mean potential independent of absolute price. Volatility_20 captures market regime.2. **The cost penalty aligns reward with actual profit.** Each trade now has a measurable cost. The agent learns that frequent position changes erode returns and that holding through trends captures gains without paying transaction costs.These two fixes are **independently necessary**: Experiment 1 had no cost issue (zero trades) but failed due to non-stationarity. Experiment 2 had good features but failed due to reward misalignment. Only when both are correct does the agent produce the desired behavior.

## Configuration: Experiment 3### Feature EngineeringWe construct a 4-dimensional observation space of stationary financial features:- **Return:** 1-day percentage change in closing price. Scale-invariant — a +1% move produces the same value at any price level. Captures short-term momentum.- **Dist_SMA_20:** Deviation from the 20-day simple moving average, normalized by the average. Measures mean-reversion potential. Positive values indicate the price is above its recent trend (overbought signal); negative values indicate below-trend pricing (oversold signal).- **Volatility_20:** 20-day rolling standard deviation of daily returns. Captures market regime — high volatility periods demand wider stop-losses and smaller position sizes.- **Rel_Volume:** Current trading volume divided by the 20-day average volume. Values above 1.0 indicate elevated trading activity, often preceding trend changes or breakout events.### Modified Reward FunctionEach position change incurs a transaction cost of 0.1% of the current asset price:$$R_t = \Delta P_t \times \text{position}_t - 0.001 \times P_t \times |\Delta\text{position}_t|$$This penalty term represents realistic trading costs (spread + commission + slippage) and aligns the agent's optimization target with actual profit and loss.

In [ ]:
print('='*60)
print('Experiment 3: Exp 3: Features + Cost')
print('='*60)

# Full feature engineering
df_f = df.copy()
df_f['Return'] = df_f['Close'].pct_change()
df_f['SMA_20'] = df_f['Close'].rolling(20).mean()
df_f['Dist_SMA_20'] = (df_f['Close'] - df_f['SMA_20']) / df_f['SMA_20']
df_f['Volatility_20'] = df_f['Return'].rolling(20).std()
df_f['Rel_Volume'] = df_f['Volume'] / df_f['Volume'].rolling(20).mean()
df_f.dropna(inplace=True)

n2 = len(df_f)
ts2 = int(n2 * 0.7)
vs2 = int(n2 * 0.85)

COST = 0.001  # 0.1% per trade

class CostAwareEnv(StocksEnv):
    def _process_data(self):
        prices = self.df['Close'].to_numpy()
        features = self.df[['Return', 'Dist_SMA_20', 'Volatility_20', 'Rel_Volume']].to_numpy()
        return prices, features
    
    def _calculate_reward(self, action):
        step_reward = super()._calculate_reward(action)
        # Penalize position changes (trades)
        new_pos = 1 if action == 1 else 0
        if self._position.value != new_pos:
            current_price = self.prices[self._current_tick]
            step_reward -= COST * current_price
        return step_reward

env_train = CostAwareEnv(df=df_f, frame_bound=(window_size, ts2), window_size=window_size)
env_val   = CostAwareEnv(df=df_f, frame_bound=(ts2, vs2), window_size=window_size)
env_test  = CostAwareEnv(df=df_f, frame_bound=(vs2, n2), window_size=window_size)

r3 = train_ppo(env_train, env_val, env_test, 'informed_trader', timesteps=25000)

## Comparative Analysis| Experiment | Trades | Reward | Profit | Failure Mode ||-----------|-------|--------|--------|-------------|| Raw Price Observations | ~0 | ~0.0 | ~1.0x | Non-stationary inputs → policy cannot generalize || Return-Based Features (No Cost) | 619 | +70 | ~0.01x | Reward misspecification → reward hacking || Stationary Features + Cost-Aware | 16 | +264.9 | **1.87x** | Both fixes applied → aligned agent |### Key Findings1. **Non-stationary state representations prevent learning entirely.** When the distribution of inputs shifts between training and deployment, the policy network fails to produce useful actions regardless of the reward function design.2. **Reward function misspecification causes misaligned optimization.** An agent with access to informative features will learn to exploit gaps between the proxy reward and the true objective, potentially destroying value while appearing to optimize well.3. **State representation and reward design are orthogonal concerns.** Each must be addressed independently. Correcting one without the other yields either an incapable agent (bad state) or a capable agent pursuing the wrong objective (bad reward).

## Generalizable Principles for Applied Reinforcement Learning### State Representation Checklist- Are the observations **stationary** across training and deployment distributions?- Do the features capture **relevant temporal structure** at appropriate scales?- Have raw values been transformed into **scale-invariant ratios or differences**?### Reward Design Checklist- Does the reward function capture **all relevant costs** (transaction costs, energy, time)?- Is there a gap between what the reward measures and what the operator actually wants?- Has the reward been tested against **adversarial exploitation** (reward hacking)?### Applicability Beyond TradingThese failure modes are not specific to financial applications. They appear in any domain where RL is deployed:| Domain | Poor State Representation | Poor Reward Design ||--------|-------------------------|-------------------|| Robotics | Encoder drift → policy fails on recalibrated hardware | Agent learns speed at cost of equipment damage || Game Playing | No frame-stacking → can't perceive motion | Agent exploits glitches instead of playing correctly || Process Control | Unnormalized sensor inputs | Controller optimizes throughput, overheating systems || Recommendation | Stale user features | Maximizes clicks over long-term engagementThe central insight: **the agent is a reflection of its environment design.** When behavior diverges from expectations, the diagnosis begins with examining what the agent observes and what it is rewarded for — not the agent itself.

In [ ]:
print('='*60)
print('FINAL COMPARISON')
print('='*60)
agents = [('Exp 1: Raw Price Observations', r1),
          ('Exp 2: No Cost Penalty', r2),
          ('Exp 3: Features + Cost', r3)]
print(f'{"Agent":35s} {"Trades":>7s} {"Reward":>8s} {"Profit":>8s}')
print('-'*60)
for name, r in agents:
    pf = r['profit']
    arrow = '✅' if pf >= 1.01 else ('⚠️' if pf >= 0.95 else '❌')
    print(f'{name:35s} {r["trades"]:>7d} {r["reward"]:>+8.1f} {pf:>7.4f}x {arrow}')
print()
print('Lessons:')
print('  1. Raw prices  →  agent struggles to learn meaningful patterns')
print('  2. No cost penalty  →  reward hacking (churning)')
print('  3. Good features + cost  →  selective, value-adding trades')